# Modeling

It is the main nootebook related to building prediction models for https://www.kaggle.com/datasets/sahistapatel96/bankadditionalfullcsv dataset.

Previous notbook related to data preprocessing and feature engineering - https://github.com/Maxstef/ml-bank-additional-project/blob/main/notebooks/02_preprocessing.ipynb


## Notebook initialization

In [1]:
# Notebook initialization
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

ROOT = Path.cwd()

while not (ROOT / "src").exists():
    ROOT = ROOT.parent

if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))

# print("Project root:", ROOT)

## Imports & Load data

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.pipeline import Pipeline
from sklearn.pipeline import make_pipeline

from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier


from src.data import load_raw_data

raw_df = load_raw_data()
raw_df.head()

,age,job,marital,education,default,housing,loan,contact,month,day_of_week,...,campaign,pdays,previous,poutcome,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed,y
0,56,housemaid,married,basic.4y,no,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
1,57,services,married,high.school,unknown,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
2,37,services,married,high.school,no,yes,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
3,40,admin.,married,basic.6y,no,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
4,56,services,married,high.school,no,no,yes,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no


In [3]:
target_col = "y"

## Split Data Train & Validation

In [4]:
from src.data import split_train_val, split_X_y

train_df, val_df = split_train_val(raw_df, stratify_col=target_col)

X_train, X_val, y_train, y_val = split_X_y(train_df, val_df, target_col)

X_train.shape, X_val.shape, y_train.shape, y_val.shape

((32950, 20), (8238, 20), (32950,), (8238,))

## Experiments Preparation & Results Tracking

In this section we set up a simple framework for running and tracking modeling experiments.

We reuse the `build_pipeline` function implemented in the [previous notebook](https://github.com/Maxstef/ml-bank-additional-project/blob/main/notebooks/02_preprocessing.ipynb) to construct configurable preprocessing and modeling pipelines.

An empty `results_df` DataFrame is created to store the results of all experiments performed in this notebook.

To automate experiment tracking, we implement an `experiment_logger` decorator that logs model parameters, pipeline configuration, and evaluation metrics (F1 and AUROC for train/test).

The `train_pipeline` function builds and fits the pipeline. Since it is decorated with `experiment_logger`, each call automatically records the experiment results in `results_df`.

In [5]:
from src.pipelining import build_pipeline
# Global experiment table
results_df = pd.DataFrame()
experiment_counter = 0

In [6]:
import time
from sklearn.metrics import f1_score, roc_auc_score

def experiment_logger(func):

    def wrapper(*args, **kwargs):
        global results_df
        global experiment_counter

        start = time.time()

        # Run training function
        pipe, X_train, X_val, y_train, y_val, pipeline_params = func(*args, **kwargs)

        fit_time = time.time() - start

        # Predictions
        y_train_pred = pipe.predict(X_train)
        y_val_pred = pipe.predict(X_val)

        y_train_prob = pipe.predict_proba(X_train)[:, 1]
        y_val_prob = pipe.predict_proba(X_val)[:, 1]

        train_f1 = f1_score(y_train, y_train_pred, pos_label="yes")
        val_f1 = f1_score(y_val, y_val_pred, pos_label="yes")

        train_auc = roc_auc_score(y_train, y_train_prob)
        val_auc = roc_auc_score(y_val, y_val_prob)

        model = pipe.named_steps["classifier"]

        # ------------------------
        # model hyperparameters
        # ------------------------
        default_model = model.__class__()
        params = model.get_params()
        default_params = default_model.get_params()

        model_params = {
            f"model__{k}": v
            for k, v in params.items()
            if k in default_params and v != default_params[k]
        }

        # ------------------------
        # pipeline params
        # ------------------------
        pipe_params = {
            f"pipe__{k}": v
            for k, v in pipeline_params.items()
            if k != "model"
        }

        # keep features count
        n_features =len(pipe.named_steps["preprocessing"].get_feature_names_out())

        row = {
            "experiment_id": experiment_counter,
            "model_name": model.__class__.__name__,
            "fit_time": fit_time,
            "train_f1": train_f1,
            "val_f1": val_f1,
            "train_auroc": train_auc,
            "val_auroc": val_auc,
            "n_features": n_features,
        }

        row.update(pipe_params)
        row.update(model_params)

        results_df = pd.concat([results_df, pd.DataFrame([row])], ignore_index=True)

        experiment_counter += 1

        return pipe

    return wrapper

In [7]:
@experiment_logger
def train_pipeline(X_train, X_val, y_train, y_val, pipeline_params):

    pipe = build_pipeline(**pipeline_params)

    pipe.fit(X_train, y_train)

    return pipe, X_train, X_val, y_train, y_val, pipeline_params

## Base Models

Use LogisticRegression, KNeighborsClassifier and DecisionTreeClassifier as base models experiments

### Baseline models
Train few baseline models with default model and pipeline params

In [8]:
base_models = [
    LogisticRegression(solver="liblinear", random_state=42),
    KNeighborsClassifier(),
    DecisionTreeClassifier(max_leaf_nodes=100, random_state=42), # Limit max_leaf_nodes=100 to avoid overfitting
]

for base_model in base_models:
    train_pipeline(X_train, X_val, y_train, y_val,
    {
        "model": base_model
    }
)

In [9]:
results_df.sort_values("val_f1", ascending=False).head()

,experiment_id,model_name,fit_time,train_f1,val_f1,train_auroc,val_auroc,n_features,model__random_state,model__solver,model__max_leaf_nodes
2,2,DecisionTreeClassifier,0.154294,0.452679,0.407924,0.795160,0.790662,59,42.0,NaN,100.0
1,1,KNeighborsClassifier,0.090787,0.488967,0.383562,0.926449,0.746241,59,NaN,NaN,NaN
0,0,LogisticRegression,0.217516,0.339418,0.337408,0.792315,0.800625,59,42.0,liblinear,NaN


<span style="background-color: #4FC3F7; display:block; padding:10px">

**Conclusions from baseline models:**

- The baseline **DecisionTreeClassifier** achieves the best **validation F1 score**, indicating strong performance on the classification objective.
- The baseline **LogisticRegression** achieves the best **validation AUROC**, which is expected since logistic regression produces well-calibrated probability estimates.
- The **KNeighborsClassifier** shows the highest scores on the **training set** but noticeably worse performance on the **validation set**, suggesting a tendency to **overfit** the training data.

</span>

### Logistic Regression

#### Feature Statistical Significance

In this section, we explore the **statistical significance of features** for the baseline Logistic Regression model using `statsmodels.Logit`.

This allows us to:

- Estimate **coefficient significance** (p-values) for each feature  
- Identify which features **increase or decrease** the probability of the target outcome  
- Gain insights into the **most influential predictors** in our model  
- Verify whether the assumptions made in the **EDA step** were correct

Features with **p-values below 0.05** are considered statistically significant.

In [10]:
pipe_lr = train_pipeline(X_train, X_val, y_train, y_val,
    {
        "poly_degree": 1,                   # no poly features
        "drop_cols": None,                  # include all cols here (except of "duration")
        "pdays_transform_mode": "binary",   # "binary" to avoid multicollinearity and all p-values 1
        "drop_cats": {                      # drop at least one value per each category to avoid multicollinearity and all p-values 1
            "job": ["unknown"],
            "marital": ["unknown"],
            "education": ["unknown", "illiterate"],
            "default": ["no"],
            "housing": ["yes"],
            "loan": ["yes"],
            "contact": ["telephone"],
            "month": ["dec"],
            "day_of_week": ["fri"],
            "poutcome": ["nonexistent"],
        },
        "model": base_models[0],            # LogisticRegression(solver="liblinear", random_state=42)
    }
)

In [11]:
X_train_transformed = pipe_lr[:-1].transform(X_train)
feature_names = pipe_lr.named_steps["preprocessing"].get_feature_names_out()
X_train_df = pd.DataFrame(
    X_train_transformed,
    columns=feature_names,
    index=X_train.index
)

In [12]:
import statsmodels.api as sm

X_train_df_const = sm.add_constant(X_train_df)
logit_model = sm.Logit(y_train.map({"no":0, "yes":1}), X_train_df_const)
stats_result_lr = logit_model.fit()

print(stats_result_lr.summary())

Optimization terminated successfully.
         Current function value: 0.277029
         Iterations 7
                           Logit Regression Results                           
Dep. Variable:                      y   No. Observations:                32950
Model:                          Logit   Df Residuals:                    32899
Method:                           MLE   Df Model:                           50
Date:                Mon, 16 Mar 2026   Pseudo R-squ.:                  0.2131
Time:                        09:52:59   Log-Likelihood:                -9128.1
converged:                       True   LL-Null:                       -11599.
Covariance Type:            nonrobust   LLR p-value:                     0.000
                                         coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------------------------------
const                                 -1.6897      0.484    

In [13]:
# round to 3 and sort by raw p-value
significance_df = pd.DataFrame({
    "feature": stats_result_lr.params.index,
    "coef": stats_result_lr.params.values.round(3),
    "p_value": stats_result_lr.pvalues.values.round(3),
    "p_value_raw": stats_result_lr.pvalues.values,
})

significance_df = significance_df.sort_values("p_value_raw")

significance_df[["feature", "coef", "p_value"]]

,feature,coef,p_value
46,numeric__emp.var.rate,-2.391,0.000
26,cat__contact_cellular,0.751,0.000
47,numeric__cons.price.idx,1.285,0.000
30,cat__month_jun,-1.264,0.000
32,cat__month_may,-0.958,0.000
33,cat__month_nov,-0.956,0.000
31,cat__month_mar,0.974,0.000
48,numeric__cons.conf.idx,0.149,0.000
40,cat__poutcome_failure,-0.392,0.000
21,cat__default_risk,-0.238,0.000


<span style="background-color: #4FC3F7; display:block; padding:10px">

**Feature Statistical Significance – Conclusions**

- Most **socio-economic context features** are statistically significant, except for `euribor3m`. This aligns with our EDA assumption that `euribor3m` is highly correlated with other socio-economic features and does not add new information to the model.  
- The features `previous` and `pdays` do not show statistical significance in the Logistic Regression model. This may be due to strong **class imbalance** (e.g., only ~4% of rows have `pdays != 999`). They might still be useful for **tree-based models**.  
- Numeric `age` does not appear significant as-is; using **binned age groups** may improve linear model performance, while tree-based models can handle raw numeric values.  
- Most **month** and **day_of_week** values are significant, suggesting **time-related effects** influence the target.  
- `housing` and `loan` remain insignificant, consistent with the EDA assumptions.  
- No statistically significant effects are observed for categorical variables `education`, `marital`, or `job` in the Logistic Regression model. This may be due to **high cardinality**.  
- Features such as `poutcome_failure`, `default_risk`, `contact_cellular`, and `campaign` are significant predictors of the target.

Overall, these results largely confirm the patterns observed in EDA and provide guidance for **feature selection and transformations** for linear models.
</span>

#### Drop Columns

Experiment with dropping additional columns for the Logistic Regression model based on the statistical significance analysis above.

In [14]:
# based on stats significans experiment above
drop_cols_options = [
    ["loan", "housing"],
    ["loan", "housing", "euribor3m"],
    ["loan", "housing", "euribor3m", "job"],
    ["loan", "housing", "euribor3m", "pdays"],
    ["loan", "housing", "euribor3m", "age"],
    ["loan", "housing", "euribor3m", "previous"],
    ["loan", "housing", "euribor3m", "job", "marital"],
    ["loan", "housing", "euribor3m", "job", "education", "marital"],
    ["loan", "housing", "euribor3m", "job", "education", "marital", "age"],
    ["loan", "housing", "euribor3m", "job", "education", "marital", "pdays"],
    ["loan", "housing", "euribor3m", "pdays", "age"],
    ["loan", "housing", "euribor3m", "pdays", "age", "previous"],
    ["loan", "housing", "euribor3m", "job", "pdays", "age", "previous"],
    ["loan", "housing", "euribor3m", "job", "education", "marital", "pdays", "previous"],
    ["loan", "housing", "euribor3m", "job", "education", "marital", "pdays", "previous", "age"],
]

for drop_cols in drop_cols_options:
    train_pipeline(X_train, X_val, y_train, y_val,
        {
            "drop_cols": drop_cols,
            "model": base_models[0],
        }
    )

In [26]:
def show_results_df(
    model_name="LogisticRegression",
    sort_by="val_f1",
    show_cols=["model_name", "fit_time", "train_f1", "val_f1", "train_auroc", "val_auroc"],
    show_count=10,
    max_experiment_id=np.inf,   # to reproduce output in case of re-run and do not show stored further experiment
):
    # Show full column content without truncation
    with pd.option_context("display.max_colwidth", None):
        display(
            results_df[
                (results_df["model_name"] == model_name) &
                (results_df["experiment_id"] <= max_experiment_id)
            ].sort_values(sort_by, ascending=False)[show_cols].head(show_count)
        )

show_results_df(
    show_cols=["model_name", "fit_time", "train_f1", "val_f1", "train_auroc", "val_auroc", "pipe__drop_cols", "n_features"],
    max_experiment_id=18
)

,model_name,fit_time,train_f1,val_f1,train_auroc,val_auroc,pipe__drop_cols,n_features
17,LogisticRegression,0.087464,0.509921,0.542418,0.933289,0.942179,"[loan, housing, euribor3m, job, education, marital, pdays, previous]",28
13,LogisticRegression,0.107964,0.509921,0.542418,0.933292,0.942181,"[loan, housing, euribor3m, job, education, marital, pdays]",29
11,LogisticRegression,0.113700,0.510652,0.542305,0.933514,0.942194,"[loan, housing, euribor3m, job, education, marital]",33
9,LogisticRegression,0.140641,0.515789,0.541333,0.934378,0.942431,"[loan, housing, euribor3m, previous]",52
12,LogisticRegression,0.104669,0.511313,0.540721,0.933504,0.942200,"[loan, housing, euribor3m, job, education, marital, age]",32
8,LogisticRegression,0.149621,0.515077,0.540613,0.934420,0.942380,"[loan, housing, euribor3m, age]",52
18,LogisticRegression,0.096360,0.509434,0.539491,0.933285,0.942190,"[loan, housing, euribor3m, job, education, marital, pdays, previous, age]",27
5,LogisticRegression,0.222543,0.514747,0.539281,0.934417,0.942376,"[loan, housing, euribor3m]",53
16,LogisticRegression,0.105706,0.511074,0.539130,0.933648,0.942146,"[loan, housing, euribor3m, job, pdays, age, previous]",36
10,LogisticRegression,0.114452,0.514806,0.538770,0.933857,0.942134,"[loan, housing, euribor3m, job, marital]",39


<span style="background-color: #4FC3F7; display:block; padding:10px">

**Feature Selection Impact – Logistic Regression**

- Excluding the features that were not statistically significant leads to a **noticeable improvement in validation metrics** for the Logistic Regression model.  
- Interestingly, including `age` still provides a **slight improvement** in F1 and AUROC on the validation set, suggesting they may carry some predictive signal even if not statistically significant in the linear model.

</span>

#### Polynomial Degree

Create polynomial degree features for the Logistic Regression experiment.

In [16]:
poly_degree_options = [1,2,3,4]

for poly_degree in poly_degree_options:
    train_pipeline(X_train, X_val, y_train, y_val,
        {
            "poly_degree": poly_degree,
            "drop_cols": ["loan", "housing", "euribor3m", "job", "education", "marital", "pdays", "previous"],
            "model": base_models[0],
        }
    )

/Users/maksymstefanko/ML/ML-love/ml-bank-additional-project/.venv/lib/python3.13/site-packages/sklearn/svm/_base.py:1258: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


In [27]:
show_results_df(
    show_cols=["model_name", "fit_time", "train_f1", "val_f1", "train_auroc", "val_auroc", "pipe__poly_degree", "n_features"],
    show_count=5,
)

,model_name,fit_time,train_f1,val_f1,train_auroc,val_auroc,pipe__poly_degree,n_features
21,LogisticRegression,7.310043,0.567655,0.587292,0.942538,0.948188,3.0,140
22,LogisticRegression,43.451722,0.587458,0.579963,0.946096,0.948698,4.0,350
20,LogisticRegression,0.397960,0.543705,0.566061,0.937656,0.945405,2.0,56
19,LogisticRegression,0.091847,0.509921,0.542418,0.933289,0.942179,1.0,28
17,LogisticRegression,0.087464,0.509921,0.542418,0.933289,0.942179,NaN,28


<span style="background-color: #4FC3F7; display:block; padding:10px">

**Polynomial Feature Expansion – Conclusions**

- The **4th-degree polynomial** does not provide any improvement in validation metrics compared to the **3rd-degree polynomial**.
- The **3rd-degree polynomial** achieves the best **validation F1 and AUROC scores**.
- The **2nd-degree polynomial** offers the best **trade-off between performance (F1/AUROC) and training time**.

</span>